In [10]:
import cv2 
import numpy as np
from IPython.display import HTML, display

In [ ]:
def show_media(*items, min_width=200):
    def box(path, title):
        ext = path.rsplit(".", 1)[-1].lower()
        if ext in ["jpg", "jpeg", "png"]:
            media = f'<img src="{path}" style="width:100%; display:block;">'
        else:
            media = f'<video style="width:100%;" autoplay loop muted playsinline><source src="{path}" type="video/mp4"></video>'
        return f"""
        <div style="flex:1; min-width:{min_width}px; padding:10px; background:white; font-family:sans-serif; text-align:center; margin:5px; box-sizing:border-box;">
            <p style="margin:0 0 8px 0; font-size:14px; color:#333;">{title}</p>
            {media}
        </div>"""
    display(HTML(f'<div style="display:flex; flex-wrap:nowrap; width:100%; background:white;">{"".join(box(p, t) for p, t in items)}</div>'))

In [12]:
Video_path = "video/video.mp4"
Bg_video_path = "video/background_video.mp4"
Bg_path = "video/B0_background.jpg"

video = cv2.VideoCapture(Video_path)
fps = int(video.get(cv2.CAP_PROP_FPS))
w = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Create background video output
fourcc = cv2.VideoWriter_fourcc(*"avc1")
bg_video_out = cv2.VideoWriter(Bg_video_path, fourcc, fps, (w, h), isColor=False)

# Buffer to store last 5 frames
frame_buffer = []
buffer_size = 5
frame_count = 0
B_0 = None

while True:
    ret, frame = video.read()
    if not ret:
        break
    
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32)
    frame_buffer.append(frame_gray)
    
    # Keep only last 5 frames
    if len(frame_buffer) > buffer_size:
        frame_buffer.pop(0)
    
    # Calculate average of frames in buffer
    if len(frame_buffer) > 0:
        bg_avg = np.mean(frame_buffer, axis=0).astype(np.uint8)
        
        # Save B_0 (first complete average after 5 frames)
        if B_0 is None and len(frame_buffer) == buffer_size:
            B_0 = bg_avg.copy()
            cv2.imwrite(Bg_path, B_0)
        
        # Write to background video
        bg_video_out.write(bg_avg)
    
    frame_count += 1

video.release()
bg_video_out.release()

# Show B_0 image, original video, and background video
show_media((Bg_path, "B_0 (Average of first 5 frames)"), 
           (Video_path, "Original Video"), 
           (Bg_video_path, "Background Video B_t"))

In [13]:
Bg_path= "video/background.jpg"
bg= cv2.imread(Bg_path)
Video_path = "video/video.mp4"

VIDEO = cv2.VideoCapture(Video_path)
w   = int(VIDEO.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(VIDEO.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(VIDEO.get(cv2.CAP_PROP_FPS))
VIDEO.release()      


show_media((Bg_path, "Background"), (Video_path, "Original Video"))

In [14]:
# threshold for pixel difference
threshold = 30

out_path = "video/change.mp4"
video = cv2.VideoCapture(Video_path)  
bg_video = cv2.VideoCapture(Bg_video_path)
out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"avc1"), fps, (w, h))
while True:
    ret, frame = video.read()
    _, bg_frame = bg_video.read()
    if not ret: break
    # Step 1 – Black & white: convert frames to grayscale
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    bg_gray = cv2.cvtColor(bg_frame, cv2.COLOR_BGR2GRAY)
    # Step 2 – Difference: absolute pixel-level difference between the two
    diff = cv2.absdiff(frame_gray, bg_gray)
    # Step 3 – Threshold: changed pixels (diff > threshold) → white, rest → black
    _, mask = cv2.threshold(diff, threshold, 255, cv2.THRESH_BINARY)
    out.write(mask)

video.release()
out.release()

show_media((Video_path, "Original Video"),(Bg_video_path, "Background video"), (out_path, "Change Detection"))


In [15]:
thresholds = [0, 10, 20, 30, 50]

bg = cv2.imread(Bg_path)
bg_gray = cv2.cvtColor(bg, cv2.COLOR_BGR2GRAY)

for t in thresholds:
    out_path = f"video/change_{t}.mp4"
    video = cv2.VideoCapture(Video_path)  
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"avc1"), fps, (w, h))
    while True:
        ret, frame = video.read()
        if not ret: break
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        diff = cv2.absdiff(frame_gray, bg_gray)
        _, mask = cv2.threshold(diff, t, 255, cv2.THRESH_BINARY)
        out.write(mask)

    video.release()
    out.release()

show_media(*[(f"video/change_{t}.mp4", f"change (threshold={t})") for t in thresholds])

In [16]:
# ── parameters you can tune ──────────────────────────────────────────
MIN_COMPONENT_PX = 50         # smaller blobs are cancelled
KERNEL_SIZE      = 5          # morphological kernel size (odd number)
ERODE_ITER       = 1          # erosion passes  (removes thin noise)
DILATE_ITER      = 2          # dilation passes (fills gaps, connects close blobs)
# ─────────────────────────────────────────────────────────────────────

# Open the existing video
change = cv2.VideoCapture("video/change_30.mp4")

# Create kernel for morphological operations
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (KERNEL_SIZE, KERNEL_SIZE))

# Setup output video writer
out_cln = cv2.VideoWriter("video/change_cleaned.mp4", cv2.VideoWriter_fourcc(*"avc1"), fps, (w, h))

while True:
    ret, frame = change.read()
    if not ret:
        break
    
    # Convert to grayscale if needed (assuming the saved video is already binary/thresholded)
    if len(frame.shape) == 3:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # ── 1. cancel small connected components ─────────────────────────
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(frame, connectivity=8)
    clean = np.zeros_like(frame)
    for lbl in range(1, n_labels):                          # 0 = background
        if stats[lbl, cv2.CC_STAT_AREA] >= MIN_COMPONENT_PX:
            clean[labels == lbl] = 255
    
    # ── 2. morphological operations ──────────────────────────────────
    clean = cv2.erode(clean, kernel, iterations=ERODE_ITER)   # remove thin noise
    clean = cv2.dilate(clean, kernel, iterations=DILATE_ITER)  # smooth & connect
    
    # Write as BGR so VideoWriter is happy
    out_cln.write(cv2.cvtColor(clean, cv2.COLOR_GRAY2BGR))

# Release resources
change.release()
out_cln.release()

show_media((Video_path, "Original Video"),("video/change_30.mp4", "Original Change Detection"),
           ("video/change_cleaned.mp4", "Cleaned Change Detection"))

In [ ]:
# Open the cleaned video
video = cv2.VideoCapture("video/change_cleaned.mp4")

# Get video properties
fps = int(video.get(cv2.CAP_PROP_FPS))
w = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Create output video with colored blobs for visualization
out_blobs = cv2.VideoWriter("video/change_blobs_visualized.mp4", 
                            cv2.VideoWriter_fourcc(*"avc1"), 
                            fps, (w, h))

frame_count = 0

while True:
    ret, frame = video.read()
    if not ret:
        break
    
    frame_count += 1
    
    # Convert to grayscale if needed
    if len(frame.shape) == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame.copy()
    
    # ── COMPUTE CONNECTED COMPONENTS (BLOBS) ────────────────────────
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        gray, connectivity=8
    )
    
    # ── VISUALIZATION ────────────────────────────────────────────────
    # Create colored visualization of blobs
    vis_frame = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    
    # Draw bounding boxes and labels for each blob (skip label 0 = background)
    for label in range(1, num_labels):
        x = stats[label, cv2.CC_STAT_LEFT]
        y = stats[label, cv2.CC_STAT_TOP]
        width = stats[label, cv2.CC_STAT_WIDTH]
        height = stats[label, cv2.CC_STAT_HEIGHT]
        area = stats[label, cv2.CC_STAT_AREA]
        cx, cy = centroids[label]
        
        # Draw bounding box (green)
        cv2.rectangle(vis_frame, (x, y), (x + width, y + height), (0, 255, 0), 2)
        
        # Draw centroid (red)
        cv2.circle(vis_frame, (int(cx), int(cy)), 3, (0, 0, 255), -1)
        
        # Add label with area
        cv2.putText(vis_frame, f"A:{area}", 
                   (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
    
    # Add frame info
    cv2.putText(vis_frame, f"Frame: {frame_count} | Blobs: {num_labels - 1}", 
               (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    out_blobs.write(vis_frame)

# Release resources
video.release()
out_blobs.release()

# ── SHOW VISUALIZATION ──────────────────────────────────────────────
show_media(
    ("video/change_cleaned.mp4", "Cleaned Change Detection"),
    ("video/change_blobs_visualized.mp4", "Connected Components (Blobs)")
)